# Modifying for W2C BOICL: Imports - This also ensures that we are working in the local repositories/python files, so changes will be reflected in this jnotebook

In [2]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt

from langchain.prompts.prompt import PromptTemplate
import copy, cloudpickle
import seaborn as sns

import sys
import os

current_dir = os.getcwd()
parent_dir = os.path.join(current_dir,'..')
sys.path.insert(0, parent_dir)

import boicl

from dotenv import load_dotenv
load_dotenv()

/Users/shane/opt/anaconda3/envs/bo_icl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

# Plotting formatt

In [3]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager
import urllib.request

urllib.request.urlretrieve(
    "https://github.com/google/fonts/raw/main/ofl/ibmplexmono/IBMPlexMono-Regular.ttf",
    "IBMPlexMono-Regular.ttf",
)
fe = font_manager.FontEntry(fname="IBMPlexMono-Regular.ttf", name="plexmono")
font_manager.fontManager.ttflist.append(fe)
plt.rcParams.update(
    {
        "axes.facecolor": "#f5f4e9",
        "grid.color": "#AAAAAA",
        "axes.edgecolor": "#333333",
        "figure.facecolor": "#FFFFFF",
        "axes.grid": False,
        "axes.prop_cycle": plt.cycler("color", plt.cm.Dark2.colors),
        "font.family": fe.name,
        "figure.figsize": (3.5, 3.5 / 1.2),
        "ytick.left": True,
        "xtick.bottom": True,
    }
)


# Dataset prep : find completion indices for tell, remove completed prompts from unlabled pool

In [ ]:
import pandas as pd
import numpy as np
import time

start_time = time.time()
# Set a consistent random seed
RANDOM_SEED = 616
np.random.seed(RANDOM_SEED)

# Enter your completed prompts and their corresponding labels gpt4o
completed_prompt_g = [""]

# completed_prompt_labels = [1.6,2.3,.34,1.9,1.6,.22,4.16]
# Enter your completed prompts and their corresponding labels gpt4.1

completed_prompt = [""]

completed_prompt_labels = []

# Load dataset
data_path = "/Users/shane/repos/BO-ICL/trimetallic_prompts_nofenicu/all_possible_prompts_nofenicu.csv"
df = pd.read_csv(data_path)

for prompt_text in completed_prompt_g:
    other = df[df["prompt"] == prompt_text]
    if not other.empty:
        idx = other.index[0]  # get the first matching index
        df.at[idx, "completion"] =  ""
        print(f"✅ Removed index {idx} with blank completion")
    else:
        print("❌ Prompt not found in the DataFrame.")

# Loop through each prompt and completion
for prompt_text, completion_value in zip(completed_prompt, completed_prompt_labels):
    match = df[df["prompt"] == prompt_text]

    if not match.empty:
        idx = match.index[0]  # get the first matching index
        df.at[idx, "completion"] = completion_value
        print(f"✅ Updated index {idx} with completion {completion_value}")
    else:
        print("❌ Prompt not found in the DataFrame.")
    

df.to_csv(data_path, index=False)

# Constants for column names
PROMPT_COL = "prompt"
COMPLETION_COL = "completion"

# Optionally shuffle the dataset
df_shuffled = df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

# Get indices where 'completion' is non-zero (not null and not just whitespace)
nonzero_completion_mask = df_shuffled[COMPLETION_COL].notna() & (df_shuffled[COMPLETION_COL].astype(str).str.strip() != "")
nonzero_completion_indices = df_shuffled.index[nonzero_completion_mask].to_numpy()

# Print first few non-zero indices for verification
print("All non-zero completion indices:", nonzero_completion_indices[:])
print(f"Total non-zero completions: {len(nonzero_completion_indices)}")

print(f"\n⏱️ Elapsed time: {time.time() - start_time:.2f} seconds")

/var/folders/gn/t7lkvwjx45l9y1jzv5lphf2h0000gn/T/ipykernel_2115/2266937687.py:28: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "completion"] =  ""


✅ Removed index 92339 with blank completion
✅ Removed index 276097 with blank completion
✅ Removed index 194866 with blank completion
✅ Removed index 290528 with blank completion
✅ Removed index 291636 with blank completion
✅ Removed index 39795 with blank completion
✅ Removed index 216062 with blank completion
✅ Updated index 92339 with completion 1.6
✅ Updated index 276097 with completion 2.3
✅ Updated index 194866 with completion 0.34
✅ Updated index 290528 with completion 1.9
✅ Updated index 75733 with completion 17.7
✅ Updated index 165882 with completion 1.75
✅ Updated index 61033 with completion 2.3
✅ Updated index 147070 with completion 0.58
All non-zero completion indices: [     1 153559 164265 187580 247740 280224 312771 318866]
Total non-zero completions: 8

⏱️ Elapsed time: 8.47 seconds


In [6]:
for i in nonzero_completion_indices:
    print(df_shuffled["completion"][i])

2.3
2.3
1.6
17.7
0.58
0.34
1.75
1.9


In [7]:
# Ensure 'completion' is numeric 
df_shuffled["completion"] = pd.to_numeric(df_shuffled["completion"], errors="coerce")

# Row with the highest completion value
index_max = df_shuffled["completion"].idxmax()
print(f"Row index with max completion: {index_max}")
print(f"Completion value: {df_shuffled.at[index_max, 'completion']}")

# If you also want to see the prompt tied to that row:
print(f"Prompt: {df_shuffled.at[index_max, 'prompt']}")

# Size of the DataFrame
print(f"DataFrame length: {len(df_shuffled)}")


Row index with max completion: 187580
Completion value: 17.7
Prompt: A trimetallic catalyst was synthesized by incipient-wetness co-impregnation onto CeO₂, with 5.0 wt % total metal loading and 0.5 wt % potassium promoter. Active metals were Zn, Mo, and W in a molar ratio 2.1:1.0:1.5. Due to differing pH requirements, sequential impregnation was employed. Initially, acidic-stable precursors were dissolved in HNO₃-acidified Milli-Q water (pH 1.8), impregnated onto the support, dried at 90 °C for 4 h, and calcined at 450 °C. Subsequently, a solution containing ammonium molybdate (para) tetrahydrate and ammonium metatungstate hydrate was used in a second impregnation step (~pH 4), followed by drying and final calcination. Prior to testing, the catalyst was reduced in CO at 600 °C under 50 psig, 20 mL/min. Testing was performed at 300 °C using a gas mixture of CO₂ (22.2 %), H₂ (66.7 %), Ar (11.1 %) at 45 mL/min and 1 atm. Sample loading was 75 mg, corresponding to GHSV 54 000 h⁻¹.
DataFram

In [8]:
system_message_path="/Users/shane/repos/BO-ICL/trimetallic_prompts_nofenicu/predict_system_message.txt"
inv_system_message_path = "/Users/shane/repos/BO-ICL/trimetallic_prompts_nofenicu/inverse_system_message.txt"


if os.path.exists(system_message_path):
    with open(system_message_path, "r") as f:
        system_message = f.read()

if os.path.exists(inv_system_message_path):
    with open(inv_system_message_path, "r") as f:
        inv_system_message = f.read()

asktell = boicl.AskTellFewShotTopk(
  prefix="",
  prompt_template=PromptTemplate(
      input_variables=["x", "y", "y_name"],
      template="Q: What is the {y_name} of {x}?@@@\nA: {y}###",
  ),
  suffix="What is the {y_name} of {x}?@@@\nA:",
  x_formatter=lambda x: f"experimental procedure: {x}",
  y_name="CO yield (%)", # label/units
  y_formatter=lambda y: f"{y:.2f}",
  model="gpt-4.1", # Do not use gpt-4 
  selector_k=5,
  temperature=0.7)

for i in nonzero_completion_indices:
    asktell.tell(df_shuffled["prompt"].iloc[i], df_shuffled["completion"].iloc[i])
    print("Told:", df_shuffled["prompt"].iloc[i], df_shuffled["completion"].iloc[i])

Told: A trimetallic catalyst was synthesized by incipient-wetness co-impregnation onto ZrO₂, with 5.0 wt % total metal loading and 0.5 wt % potassium promoter. Active metals were Zn, Zr, and Mo in a molar ratio 1.5:1.0:6.7. Due to differing pH requirements, sequential impregnation was employed. Initially, acidic-stable precursors were dissolved in HNO₃-acidified Milli-Q water (pH 1.8), impregnated onto the support, dried at 90 °C for 4 h, and calcined at 450 °C. Subsequently, a solution containing ammonium molybdate (para) tetrahydrate was used in a second impregnation step (~pH 4), followed by drying and final calcination. Prior to testing, the catalyst was reduced in CO at 300 °C under 50 psig, 20 mL/min. Testing was performed at 300 °C using a gas mixture of CO₂ (22.2 %), H₂ (66.7 %), Ar (11.1 %) at 45 mL/min and 1 atm. Sample loading was 75 mg, corresponding to GHSV 54 000 h⁻¹. 2.3
Told: A trimetallic catalyst was synthesized by incipient-wetness co-impregnation onto CeO₂, with 5.0

# Remove Ran Experiments

In [9]:
raw_data = df_shuffled.drop(nonzero_completion_indices)
print("New data set size is :", len(raw_data))

New data set size is : 359992


In [10]:
# import importlib, boicl.pool
# importlib.reload(boicl.pool)
# from boicl.pool import Pool


# pool_path = "no_FeNiCu_trimetallics.pkl"

# if os.path.exists(pool_path):
#   with open(pool_path, "rb") as f:
#     pool = cloudpickle.load(f)
#   pool.reset()
# else:
#   x = [raw_data[PROMPT_COL].iloc[i] for i in raw_data.index]
#   pool = boicl.Pool(list(x), formatter=lambda x: f"experimental procedure: {x}")
#   cloudpickle.dump(pool, open(pool_path, "wb"))
# from boicl.pool import Pool      # ← add this line (or reload if you just edited the file)

# pool_path = "no_FeNiCu_trimetallics.pkl"

# import cloudpickle, os

# if os.path.exists(pool_path):
#     with open(pool_path, "rb") as f:
#         pool = cloudpickle.load(f)
#     pool.reset()
# else:
#     prompts = raw_data[PROMPT_COL].tolist()
#     pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")
#     with open(pool_path, "wb") as f:
#         cloudpickle.dump(pool, f)

from boicl.pool import Pool        # the new version
import cloudpickle, os, numpy as np, faiss, pickle

prompts = raw_data[PROMPT_COL].tolist()          # your full prompt list

# ---- A. build the pool (slow, calls OpenAI) -----------------------
pool = Pool(prompts, formatter=lambda x: f"experimental procedure: {x}")

# ---- B. export the 3 lightweight files once -----------------------
np.save("emb.npy", pool._emb_matrix)             # ~4 GB
faiss.write_index(pool.index, "index.faiss")     # ~4 GB
with open("prompts.pkl", "wb") as f:
    pickle.dump(pool._pool, f)

print("✅ Pre-built files written.")



✅ Loaded cache (341,998 embeddings).
✅ Pre-built files written.


In [13]:
import importlib, boicl.pool
importlib.reload(boicl.pool)
from boicl.pool import Pool

pool = Pool.from_prebuilt(
    prompts_path="prompts.pkl",
    emb_path="emb.npy",
    index_path="index.faiss",
    formatter=lambda x: f"experimental procedure: {x}",
)

len(pool)

359992

In [11]:
import pandas as pd
df_noFe = pd.read_csv("/Users/shane/repos/BO-ICL/trimetallic_prompts_nofenicu/all_possible_prompts.csv")

[new_prompt],[aq_value],[mean],[std]=asktell.ask(pool,aq_fxn="expected_improvement",system_message=system_message, inv_system_message=inv_system_message)

df_noFe.iloc[df_noFe[df_noFe["prompt"] == new_prompt].index[0]],print(
        f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
        f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
        f"\033[1mStd dev:\033[0m {std:.3f}"
    )

Acquisition value: 0.000
Mean prediction: 2.310
Std dev: 0.971


(M1                                                              Zn
 M2                                                              Mo
 M3                                                              Mn
 A_mol                                                       0.5455
 B_mol                                                       0.2727
 C_mol                                                       0.1818
 mol_ratio                                              3.0:1.5:1.0
 support                                                       TiO₂
 promoter                                                       NaN
 pretreat_gas                                                    H₂
 pretreat_temp                                                550.0
 prompt           A trimetallic catalyst was synthesized by inci...
 Name: 42830, dtype: object,
 None)

In [12]:
[new_prompt]

['A trimetallic catalyst was synthesized by incipient-wetness co-impregnation onto TiO₂, with 5.0 wt % total metal loading. Active metals were Zn, Mo, and Mn in a molar ratio 3.0:1.5:1.0. Due to differing pH requirements, sequential impregnation was employed. Initially, acidic-stable precursors were dissolved in HNO₃-acidified Milli-Q water (pH 1.8), impregnated onto the support, dried at 90 °C for 4 h, and calcined at 450 °C. Subsequently, a solution containing ammonium molybdate (para) tetrahydrate was used in a second impregnation step (~pH 4), followed by drying and final calcination. Prior to testing, the catalyst was reduced in H₂ at 550 °C under 50 psig, 20 mL/min. Testing was performed at 300 °C using a gas mixture of CO₂ (22.2 %), H₂ (66.7 %), Ar (11.1 %) at 45 mL/min and 1 atm. Sample loading was 75 mg, corresponding to GHSV 54 000 h⁻¹.']

# Trajectory 2

In [26]:
import pandas as pd
df_noFe = pd.read_csv("/Users/shane/repos/BO-ICL/trimetallic_prompts_nofenicu/all_possible_prompts.csv")

[new_prompt],[aq_value],[mean],[std]=asktell.ask(pool,aq_fxn="expected_improvement",system_message=system_message, inv_system_message=inv_system_message)

df_noFe.iloc[df_noFe[df_noFe["prompt"] == new_prompt].index[0]],print(
        f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
        f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
        f"\033[1mStd dev:\033[0m {std:.3f}"
    )

Acquisition value: 3.440
Mean prediction: 20.220
Std dev: 4.754


(M1                                                              Zn
 M2                                                              Mo
 M3                                                              Mn
 A_mol                                                       0.9091
 B_mol                                                       0.0182
 C_mol                                                       0.0727
 mol_ratio                                             50.0:1.0:4.0
 support                                                       CeO₂
 promoter                                                 potassium
 pretreat_gas                                                    CO
 pretreat_temp                                                550.0
 prompt           A trimetallic catalyst was synthesized by inci...
 Name: 90882, dtype: object,
 None)

In [21]:
import pandas as pd
df_noFe = pd.read_csv("/Users/shane/repos/BO-ICL/trimetallic_prompts_nofenicu/all_possible_prompts.csv")

[new_prompt],[aq_value],[mean],[std]=asktell.ask(pool,aq_fxn="greedy",system_message=system_message, inv_system_message=inv_system_message)

df_noFe.iloc[df_noFe[df_noFe["prompt"] == new_prompt].index[0]],print(
        f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
        f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
        f"\033[1mStd dev:\033[0m {std:.3f}"
    )

Acquisition value: 11.400
Mean prediction: 10.130
Std dev: 2.496


(M1                                                              Zn
 M2                                                              Zr
 M3                                                              Mo
 A_mol                                                       0.4545
 B_mol                                                       0.2182
 C_mol                                                       0.3273
 mol_ratio                                              2.1:1.0:1.5
 support                                                       ZrO₂
 promoter                                                 potassium
 pretreat_gas                                                    H₂
 pretreat_temp                                                450.0
 prompt           A trimetallic catalyst was synthesized by inci...
 Name: 75093, dtype: object,
 None)

In [17]:
new_prompt

'A trimetallic catalyst was synthesized by incipient-wetness co-impregnation onto CeO₂, with 5.0 wt % total metal loading and 0.5 wt % potassium promoter. Active metals were Zn, Mo, and W in a molar ratio 2.1:1.0:1.5. Due to differing pH requirements, sequential impregnation was employed. Initially, acidic-stable precursors were dissolved in HNO₃-acidified Milli-Q water (pH 1.8), impregnated onto the support, dried at 90 °C for 4 h, and calcined at 450 °C. Subsequently, a solution containing ammonium molybdate (para) tetrahydrate and ammonium metatungstate hydrate was used in a second impregnation step (~pH 4), followed by drying and final calcination. Prior to testing, the catalyst was reduced in CO at 600 °C under 50 psig, 20 mL/min. Testing was performed at 300 °C using a gas mixture of CO₂ (22.2 %), H₂ (66.7 %), Ar (11.1 %) at 45 mL/min and 1 atm. Sample loading was 75 mg, corresponding to GHSV 54 000 h⁻¹.'

In [17]:
import pandas as pd
df_noFe = pd.read_csv("/Users/shane/repos/BO-ICL/trimetallic_prompts_nofenicu/all_possible_prompts.csv")

[new_prompt],[aq_value],[mean],[std]=asktell.ask(pool,aq_fxn="expected_improvement",system_message=system_message, inv_system_message=inv_system_message)

df_noFe.iloc[df_noFe[df_noFe["prompt"] == new_prompt].index[0]],print(
        f"\033[1mAcquisition value:\033[0m {aq_value:.3f}\n"
        f"\033[1mMean prediction:\033[0m {mean:.3f}\n"
        f"\033[1mStd dev:\033[0m {std:.3f}"
    )

Acquisition value: 0.000
Mean prediction: 0.698
Std dev: 0.204


(M1                                                              Zn
 M2                                                              Mo
 M3                                                               W
 A_mol                                                       0.1818
 B_mol                                                       0.0909
 C_mol                                                       0.7273
 mol_ratio                                              2.0:1.0:8.0
 support                                                       CeO₂
 promoter                                                 potassium
 pretreat_gas                                                    CO
 pretreat_temp                                                550.0
 prompt           A trimetallic catalyst was synthesized by inci...
 Name: 237732, dtype: object,
 None)